In [1]:
import dask
from dask.distributed import Client, LocalCluster

dask.config.config["distributed"]["dashboard"]["link"] = "{JUPYTERHUB_SERVICE_PREFIX}proxy/{host}:{port}/status"

client = Client(threads_per_worker=4, n_workers=16)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/amahesh/perlmutter-exclusive-node-cpu/proxy/127.0.0.1:8787/status,
Dashboard: /user/amahesh/perlmutter-exclusive-node-cpu/proxy/127.0.0.1:8787/status,Workers: 16
Total threads: 64,Total memory: 476.37 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:45343,Workers: 16
Dashboard: /user/amahesh/perlmutter-exclusive-node-cpu/proxy/127.0.0.1:8787/status,Total threads: 64
Started: Just now,Total memory: 476.37 GiB
Comm: tcp://127.0.0.1:36261,Total threads: 4
Dashboard: /user/amahesh/perlmutter-exclusive-node-cpu/proxy/127.0.0.1:39361/status,Memory: 29.77 GiB
Nanny: tcp://127.0.0.1:36567,


In [2]:
import xarray as xr
import numpy as np
from datetime import datetime
from calendar import monthrange

In [3]:
def _get_sfc_path_pm(path: str, time, var: str) -> str:
    SFC_VAR_TO_CODE_MAP = {
        "VAR_10U": "128_165_10u",
        "VAR_10V": "128_166_10v",
        "sp": "128_134_sp",
        "tcwv": "128_137_tcwv",
        "VAR_2T": "128_167_2t",
        "VAR_2D" : "128_168_2d",
        "msl": "128_151_msl",
        "VAR_100U": "228_246_100u",
        "VAR_100V": "228_247_100v",
    }

    days = monthrange(time.year, time.month)[1]
    code = SFC_VAR_TO_CODE_MAP[var]
    return "{}/{}{:02}/e5.oper.an.sfc.{}.ll025sc.{}{:02}0100_{}{:02}{:02}23.nc".format(
        path,
        time.year,
        time.month,
        code,
        time.year,
        time.month,
        time.year,
        time.month,
        days,
    )

def _get_pressure_path_pm(path: str, time, var: str) -> str:
    PRES_VAR_TO_CODE_MAP = {
        "u": "128_131_u.ll025uv",
        "v": "128_132_v.ll025uv",
        "z": "128_129_z.ll025sc",
        "t": "128_130_t.ll025sc",
        "r": "128_157_r.ll025sc",
    }
    code = PRES_VAR_TO_CODE_MAP[var]
    return "{}/{}{:02}/e5.oper.an.pl.{}.{}{:02}{:02}00_{}{:02}{:02}23.nc".format(
        path,
        time.year,
        time.month,
        code,
        time.year,
        time.month,
        time.day,
        time.year,
        time.month,
        time.day,
    )

In [ ]:
lookup_table = xr.open_zarr("/pscratch/sd/a/amahesh/hens/prod_heat_index_lookup.zarr").load()

In [4]:
def _saturation_vapor_pressure(temperature):
    """
    temperature must be in units of Kelvin
    """
    sat_pressure_0c = 6.112
    return sat_pressure_0c * np.exp(17.67 * (temperature - 273.15) / (temperature - 29.65))

def _calculate_rh_from_dewpoint(t2m, d2m):
    """
    t2m: temperature at 2m in Kelvin
    d2m: dewpoint at 2m in Kelvin
    """
    sat_vapor_pressure = _saturation_vapor_pressure(t2m)
    vapor_pressure = _saturation_vapor_pressure(d2m)
    return vapor_pressure / sat_vapor_pressure

In [5]:
M3522 = "/dvs_ro/cfs/cdirs/m3522/cmip6/ERA5/"

def open_t850(month, hour):
    paths = []
    base_path = M3522 + "/e5.oper.an.pl/"
    for year in range(1993,2017):
        for day in range(1, monthrange(year, month)[1]+1):
            paths.append(_get_pressure_path_pm(base_path, datetime(year, month, day), 't'))
    ds = xr.open_mfdataset(paths).sel(level=850, drop=True)
    return ds.sel(time=ds['time.hour'] == hour)

def open_t2m(month, hour):
    paths = []
    base_path = M3522 + "/e5.oper.an.sfc/"
    for year in range(1993,2017):
        paths.append(_get_sfc_path_pm(base_path, datetime(year, month, 1), 'VAR_2T'))
    ds = xr.open_mfdataset(paths)
    return ds.sel(time=ds['time.hour'] == hour)

def open_wind_speed10m(month, hour):
    paths_u10m, paths_v10m = [], []
    base_path = M3522 + "/e5.oper.an.sfc/"
    for year in range(1993,2017):
        paths_u10m.append(_get_sfc_path_pm(base_path, datetime(year, month, 1), 'VAR_10U'))
        paths_v10m.append(_get_sfc_path_pm(base_path, datetime(year, month, 1), 'VAR_10V'))
    ds = xr.open_mfdataset(paths_u10m)
    ds_v10m = xr.open_mfdataset(paths_v10m)
    ds['wind_speed10m'] = np.sqrt(ds['VAR_10U'] ** 2 + ds_v10m['VAR_10V']**2)
    return ds[['wind_speed10m']].sel(time=ds['time.hour'] == hour)

def open_heat_index(month, hour):
    paths_t2m, paths_d2m = [], []
    M3522 = "/dvs_ro/cfs/cdirs/m3522/cmip6/ERA5/"
    base_path = M3522 + "/e5.oper.an.sfc/"
    outputs = []
    for year in range(1993,2017):
        paths_t2m = _get_sfc_path_pm(base_path, datetime(year, month, 1), 'VAR_2T')
        paths_d2m = _get_sfc_path_pm(base_path, datetime(year, month, 1), 'VAR_2D')
        ds = xr.open_mfdataset(paths_t2m)
        ds = ds.sel(time=ds['time.hour'] == hour)
        ds_d2m = xr.open_mfdataset(paths_d2m)
        ds_d2m = ds_d2m.sel(time=ds_d2m['time.hour'] == hour)
        ds['VAR_2T'] = ds['VAR_2T'].load()
        rh = _calculate_rh_from_dewpoint(ds['VAR_2T'], 
                                     ds_d2m['VAR_2D']).load()
        outputs.append(lookup_table.sel(Rh=rh, Ta=ds['VAR_2T'], method='nearest'))
        print(year)
    return xr.concat(outputs, dim='time')


In [7]:
hi = open_heat_index(3,0)
hi

<xarray.DataArray (time: 31, latitude: 721, longitude: 1440)>
array([[[0.7694661 , 0.7694661 , 0.7694661 , ..., 0.7694661 ,
         0.7694661 , 0.7694661 ],
        [0.77090955, 0.7709133 , 0.77104974, ..., 0.7706221 ,
         0.77062917, 0.7707695 ],
        [0.7912197 , 0.79149497, 0.79149836, ..., 0.79078263,
         0.79093164, 0.7910774 ],
        ...,
        [0.7760718 , 0.7760681 , 0.7760649 , ..., 0.77549756,
         0.77564037, 0.77592576],
        [0.77764183, 0.7777811 , 0.7777811 , ..., 0.777499  ,
         0.777499  , 0.77764183],
        [0.77014244, 0.77014244, 0.77014244, ..., 0.77014244,
         0.77014244, 0.77014244]],

       [[0.7033211 , 0.7033211 , 0.7033211 , ..., 0.7033211 ,
         0.7033211 , 0.7033211 ],
        [0.7037519 , 0.7037519 , 0.7037519 , ..., 0.7037519 ,
         0.7037519 , 0.7037519 ],
        [0.70212764, 0.70212764, 0.70212764, ..., 0.7021322 ,
         0.7019991 , 0.7019991 ],
...
        [0.7196501 , 0.7196501 , 0.7195167 , ..., 0.7200764 ,
         0.7199342 , 0.7197922 ],
        [0.7241248 , 0.7239908 , 0.7239908 , ..., 0.7244061 ,
         0.7242676 , 0.72412926],
        [0.7422241 , 0.7422241 , 0.7422241 , ..., 0.7422241 ,
         0.7422241 , 0.7422241 ]],

       [[0.7009351 , 0.7009351 , 0.7009351 , ..., 0.7009351 ,
         0.7009351 , 0.7009351 ],
        [0.70334995, 0.70334995, 0.7032252 , ..., 0.70360404,
         0.7034793 , 0.70335454],
        [0.7119445 , 0.71181446, 0.71156275, ..., 0.7123351 ,
         0.7122049 , 0.7120747 ],
        ...,
        [0.7374223 , 0.7374141 , 0.73740166, ..., 0.7375808 ,
         0.73743516, 0.73743075],
        [0.71768796, 0.7176791 , 0.7175359 , ..., 0.71783566,
         0.71783096, 0.7178267 ],
        [0.73708284, 0.73708284, 0.73708284, ..., 0.73708284,
         0.73708284, 0.73708284]]], dtype=float32)
Coordinates:
  * latitude   (latitude) float64 90.0 89.75 89.5 89.25 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float64 0.0 0.25 0.5 0.75 ... 359.0 359.2 359.5 359.8
  * time       (time) datetime64[ns] 1993-01-01 1993-01-02 ... 1993-01-31

In [10]:
t2m = open_t2m(3,0)
t2m

<xarray.Dataset>
Dimensions:    (time: 744, latitude: 721, longitude: 1440)
Coordinates:
  * latitude   (latitude) float64 90.0 89.75 89.5 89.25 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float64 0.0 0.25 0.5 0.75 ... 359.0 359.2 359.5 359.8
  * time       (time) datetime64[ns] 1993-03-01 1993-03-02 ... 2016-03-31
Data variables:
    VAR_2T     (time, latitude, longitude) float32 dask.array<chunksize=(31, 721, 1440), meta=np.ndarray>
    utc_date   (time) int32 dask.array<chunksize=(31,), meta=np.ndarray>
Attributes:
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB1 data to netCDF4.
    NETCDF_VERSION:       4.6.3
    CONVERSION_PLATFORM:  Linux r14i2n18 4.12.14-94.41-default #1 SMP Wed Oct...
    CONVERSION_DATE:      Fri Jul  5 23:26:59 MDT 2019
    Conventions:          CF-1.6
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    history:              Fri Jul  5 23:27:15 2019: ncks -4 --ppc default=7 e...
    NCO:                  netCDF Operators version 4.7.9 (Homepage = http://n...

In [7]:
curr_variable = 't2m'

for month in range(1,13):
    for hour in [0,6,12,18]:
        t2m = open_t2m(month, hour)
        t2m = t2m.chunk({'time' : -1, 'latitude' : 20, 'longitude' : 40})
        t2m.mean('time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds_cold/{}_mean_{:02d}_{:02d}".format(curr_variable, month,hour))
        t2m.std('time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds_cold/{}_std_{:02d}_{:02d}".format(curr_variable, month,hour))
        t2m.quantile(0.05, dim='time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds_cold/{}_percentile05_{:02d}_{:02d}".format(curr_variable, month,hour))
        t2m.quantile(0.01, dim='time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds_cold/{}_percentile01_{:02d}_{:02d}".format(curr_variable, month,hour))
        t2m.quantile(0.001, dim='time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds_cold/{}_percentile0p1_{:02d}_{:02d}".format(curr_variable, month,hour))
        t2m.min(dim='time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds_cold/{}_min_{:02d}_{:02d}".format(curr_variable, month,hour))
        
        print("completed {:02d} {:02d}".format(month, hour))

completed 01 00
completed 01 06
completed 01 12
completed 01 18
completed 02 00
completed 02 06
completed 02 12
completed 02 18
completed 03 00
completed 03 06
completed 03 12
completed 03 18
completed 04 00
completed 04 06
completed 04 12
completed 04 18
completed 05 00
completed 05 06
completed 05 12
completed 05 18
completed 06 00
completed 06 06
completed 06 12
completed 06 18
completed 07 00
completed 07 06
completed 07 12
completed 07 18
completed 08 00
completed 08 06
completed 08 12
completed 08 18
completed 09 00
completed 09 06
completed 09 12
completed 09 18
completed 10 00
completed 10 06
completed 10 12
completed 10 18
completed 11 00
completed 11 06
completed 11 12
completed 11 18
completed 12 00
completed 12 06
completed 12 12
completed 12 18


In [13]:
def open_t2m_daily(month, mode):
    paths = []
    base_path = M3522 + "/e5.oper.an.sfc/"
    for year in range(1993,2017):
        paths.append(_get_sfc_path_pm(base_path, datetime(year, month, 1), 'VAR_2T'))
    ds = xr.open_mfdataset(paths)
    ds = ds.sel(time=np.in1d(ds['time.hour'], [0,6,12,18]))
    # ds = ds.chunk({'time' : -1, 'latitude' : 10, 'longitude' : 20})
    if mode == 'mean':
        return ds.resample(time='D').mean()
    else:
        print('resampling to daily max')
        return ds.resample(time='D').max()

In [14]:
for month in [1,2,3,4,5,10,11,12]:##[6, 7,8,9]:
    for mode in ['mean', 'max']:
        t2m = open_t2m_daily(month, mode)
        t2m = t2m.chunk({'time' : -1})
        t2m.quantile(0.95, dim='time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds_daily/t2m_percentile95_{:02d}_daily{}.zarr".format(month,mode), mode='w')
        t2m.mean('time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds_daily/t2m_mean_{:02d}_daily{}.zarr".format(month,mode), mode='w')
        t2m.std('time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds_daily/t2m_std_{:02d}_daily{}.zarr".format(month,mode), mode='w')
        t2m.quantile(0.99, dim='time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds_daily/t2m_percentile99_{:02d}_daily{}.zarr".format(month,mode), mode='w')
        t2m.quantile(0.999, dim='time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds_daily/t2m_percentile99p9_{:02d}_daily{}.zarr".format(month,mode), mode='w')
        t2m.max(dim='time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds_daily/t2m_max_{:02d}_daily{}.zarr".format(month,mode), mode='w')
        
        print("completed {:02d} {}".format(month, mode))

/opt/miniconda3/lib/python3.9/site-packages/xarray/core/indexing.py:1443: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]


completed 01 mean
resampling to daily max


/opt/miniconda3/lib/python3.9/site-packages/xarray/core/indexing.py:1443: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]


completed 01 max


/opt/miniconda3/lib/python3.9/site-packages/xarray/core/indexing.py:1443: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]


...

completed 11 max


/opt/miniconda3/lib/python3.9/site-packages/xarray/core/indexing.py:1443: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]


completed 12 mean
resampling to daily max


/opt/miniconda3/lib/python3.9/site-packages/xarray/core/indexing.py:1443: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]


completed 12 max


In [6]:
curr_variable = 'wind_speed10m'

for month in range(1,6):
    for hour in [0,6,12,18]:
        t2m = open_wind_speed10m(month, hour)
        t2m = t2m.chunk({'time' : -1, 'latitude' : 20, 'longitude' : 40})
        t2m.mean('time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds/{}_mean_{:02d}_{:02d}".format(curr_variable, month,hour),
                                mode='w')
        t2m.std('time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds/{}_std_{:02d}_{:02d}".format(curr_variable, month,hour),
                                mode='w')
        t2m.quantile(0.95, dim='time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds/{}_percentile95_{:02d}_{:02d}".format(curr_variable, month,hour),
                                mode='w')
        t2m.quantile(0.99, dim='time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds/{}_percentile99_{:02d}_{:02d}".format(curr_variable, month,hour),
                                mode='w')
        t2m.quantile(0.999, dim='time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds/{}_percentile99p9_{:02d}_{:02d}".format(curr_variable, month,hour),
                                mode='w')
        t2m.max(dim='time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds/{}_max_{:02d}_{:02d}".format(curr_variable, month,hour),
                                mode='w')
        
        print("completed {:02d} {:02d}".format(month, hour))

/opt/miniconda3/lib/python3.9/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()


completed 01 06
completed 01 12
completed 01 18
completed 02 00
completed 02 06
completed 02 12
completed 02 18
completed 03 00
completed 03 06
completed 03 12
completed 03 18
completed 04 00
completed 04 06
completed 04 12
completed 04 18


2025-03-22 00:34:21,401 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 24.10 GiB -- Worker memory limit: 29.77 GiB
2025-03-22 00:34:21,464 - distributed.worker.memory - WARNING - Worker is at 24% memory usage. Resuming worker. Process memory: 7.19 GiB -- Worker memory limit: 29.77 GiB
2025-03-22 00:36:04,464 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 23.86 GiB -- Worker memory limit: 29.77 GiB
2025-03-22 00:36:04,678 - distributed.worker.memory - WARNING - Worker is at 25% memory usage. Resuming worker. Process memory: 7.66 GiB -- Worker memory limit: 29.77 GiB


completed 05 00
completed 05 06
completed 05 12
completed 05 18


In [ ]:
curr_variable = 'heat_index'

for month in range(6,10):
    for hour in [0,6,12,18]:
        t2m = open_heat_index(month, hour)
        # t2m = t2m.chunk({'time' : -1, 'latitude' : 20, 'longitude' : 40})
        t2m.mean('time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds/{}_mean_{:02d}_{:02d}".format(curr_variable, month,hour),
                                mode='w')
        t2m.std('time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds/{}_std_{:02d}_{:02d}".format(curr_variable, month,hour),
                                mode='w')
        t2m.quantile(0.95, dim='time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds/{}_percentile95_{:02d}_{:02d}".format(curr_variable, month,hour),
                                mode='w')
        t2m.quantile(0.99, dim='time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds/{}_percentile99_{:02d}_{:02d}".format(curr_variable, month,hour),
                                mode='w')
        t2m.quantile(0.999, dim='time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds/{}_percentile99p9_{:02d}_{:02d}".format(curr_variable, month,hour),
                                mode='w')
        t2m.max(dim='time').to_zarr("/pscratch/sd/a/amahesh/hens/thresholds/{}_max_{:02d}_{:02d}".format(curr_variable, month,hour),
                                mode='w')
        
        print("completed {:02d} {:02d}".format(month, hour))